In [1]:
import pandas as pd
import time 
import requests
import json
#other requirements: sqlalchemy, pscycopg2

In [2]:
from DatabaseCreator import DatabaseCreator

In [3]:
start=time.perf_counter()

In [4]:
current_season = 20252026

In [5]:
# #get fixtures before changing code
# from Load_Merge_FPL_Data import FPLDataManager 
# import CJDH_local_settings

# #Run Database Creator
# if __name__ == "__main__":
#     manager = FPLDataManager()
#     #FPLAPIDataFetcher: id max = 5 - remove after testing
#     fpl_data, fixtures_with_elo, merged, current_year_players_allfixtures = manager.load_2025_26_season_from_api()

# current_year_players_allfixtures.head()

In [6]:
from Load_Merge_FPL_Data import FPLDataManager
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    manager = FPLDataManager()
    df_all_seasons = manager.load_all_seasons()
    print(f"\nTotal records: {len(df_all_seasons)}")

    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

    #Add a new staging table & data into the database
    playergw = df_all_seasons[['player_name_id','element','value','season','event','fixture','total_points',
                            'minutes','goals_scored','assists','team_elo','opp_team_elo','position','bonus','bps',
                            'clean_sheets','goals_conceded','was_home','expected_assists','expected_goal_involvements',
                            'expected_goals','expected_goals_conceded','starts',
                            'cbi','defensive_contribution','recoveries','tackles',
                            'saves','team_name','opp_team_name']]

    table_name = "playergw"
    db_creator.create_staging_table_then_insert_data(table_name, data=playergw)
    playergwdf = db_creator.table_to_df(table_name=table_name)
    print("Final playergw dataframe loaded into psql!")
    print(playergwdf.info())

Loading season 20182019...
Loading season 20192020...
Loading season 20202021...
Loading season 20212022...
Loading season 20222023...
Loading season 20232024...
Loading season 2024-25 (manual merge)...
Loading season 2025-26 (API)...
Loaded data for element_id: 1/834
Loaded data for element_id: 2/834
Loaded data for element_id: 3/834
Loaded data for element_id: 4/834
Loaded data for element_id: 5/834
Loaded data for element_id: 6/834
Loaded data for element_id: 7/834
Loaded data for element_id: 8/834
Loaded data for element_id: 9/834
Loaded data for element_id: 10/834
Loaded data for element_id: 11/834
Loaded data for element_id: 12/834
Loaded data for element_id: 13/834
Loaded data for element_id: 14/834
Loaded data for element_id: 15/834
Loaded data for element_id: 16/834
Loaded data for element_id: 17/834
Loaded data for element_id: 18/834
Loaded data for element_id: 19/834
Loaded data for element_id: 20/834
Loaded data for element_id: 21/834
Loaded data for element_id: 22/834
Load

In [7]:
filtered = (playergwdf['season']==20252026) & (playergwdf['event']==36) & (playergwdf['player_name_id']=='Erling Haaland')
playergwdf[filtered].head()

,player_name_id,element,value,season,event,fixture,total_points,minutes,goals_scored,assists,...,expected_goals,expected_goals_conceded,starts,cbi,defensive_contribution,recoveries,tackles,saves,team_name,opp_team_name
193456,Erling Haaland,430.0,147.0,20252026.0,36.0,356.0,11.0,90.0,1.0,1.0,...,1.22,0.24,1.0,0.0,1.0,1.0,0.0,0.0,Man City,Brentford
193457,Erling Haaland,430.0,147.0,20252026.0,36.0,307.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,Man City,Crystal Palace


In [8]:
# import CJDH_local_settings

# #Run Database Creator
# if __name__ == "__main__":
#     db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
#     fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

In [9]:
from FPL_League_Data_Scraper import FPL_League_Data_Scraper

#Run Data Scraper for TRDL div1 league
league_code = "1447854"


if __name__ == "__main__":
    trdl = FPL_League_Data_Scraper(league_code, current_season)
    picksdf, gw_summary = trdl.scrape_picks()

    picksdf['div'] = 1
    gw_summary['div'] = 1

    #Add a new staging table & data into the database
    picks = picksdf[['member', 'memberid','season','div', 'gw', 'pick', 
                    'element', 'position', 'multiplier', 
                    'is_captain', 'is_vice_captain']]
    
    table_name = "fpl_picks"
    db_creator.create_staging_table_then_insert_data(table_name, data=picks)
    fpl_picks = db_creator.table_to_df(table_name=table_name)

    table_name = "gw_summary"
    db_creator.create_staging_table_then_insert_data(table_name, data=gw_summary)
    gw_summarydf = db_creator.table_to_df(table_name=table_name)
    print(gw_summarydf.info())

Data fetched, league name:  Riverside Divisional League 1
Fetched history for Christopher Harris (ID: 145486)
Fetched history for Tommy Davies (ID: 824703)
Fetched history for Tajwar Shelim (ID: 4773459)
Fetched history for Christakis Christodoulou (ID: 4774874)
Fetched history for Oliver Carroll (ID: 4772822)
Fetched history for Ryan Boyce (ID: 4933105)
Fetched history for Callum Harling (ID: 2506220)
Max gameweek data available for is:  37
145486 Christopher Harris GW 1 Pick 1
145486 Christopher Harris GW 1 Pick 2
145486 Christopher Harris GW 1 Pick 3
145486 Christopher Harris GW 1 Pick 4
145486 Christopher Harris GW 1 Pick 5
145486 Christopher Harris GW 1 Pick 6
145486 Christopher Harris GW 1 Pick 7
145486 Christopher Harris GW 1 Pick 8
145486 Christopher Harris GW 1 Pick 9
145486 Christopher Harris GW 1 Pick 10
145486 Christopher Harris GW 1 Pick 11
145486 Christopher Harris GW 1 Pick 12
145486 Christopher Harris GW 1 Pick 13
145486 Christopher Harris GW 1 Pick 14
145486 Christophe

In [10]:
#Run Data Scraper for TRDL div2 league
league_code2 = "1032210"


if __name__ == "__main__":
    trdl2 = FPL_League_Data_Scraper(league_code2, current_season)
    picksdf2, gw_summary2 = trdl2.scrape_picks()

    picksdf2['div'] = 2
    gw_summary2['div'] = 2

    #Add a new staging table & data into the database
    picks2 = picksdf2[['member', 'memberid','season','div', 'gw', 'pick', 
                    'element', 'position', 'multiplier', 
                    'is_captain', 'is_vice_captain']]
    

    table_name2 = "fpl_picks_2"
    db_creator.create_staging_table_then_insert_data(table_name2, data=picks2)
    fpl_picks2 = db_creator.table_to_df(table_name=table_name2)
    print(fpl_picks2.info())

    table_name = "gw_summary_2"
    db_creator.create_staging_table_then_insert_data(table_name, data=gw_summary2)
    gw_summarydf2 = db_creator.table_to_df(table_name=table_name)
    print(gw_summarydf2.info())

Data fetched, league name:  TRDL 2 25/26
Fetched history for Ben Shankland (ID: 4771338)
Fetched history for Rhys Birdy (ID: 4899844)
Fetched history for Jack Carroll (ID: 4772682)
Fetched history for Conner Harling (ID: 4774822)
Fetched history for Christopher Shankland (ID: 4771364)
Fetched history for Anthony Collier (ID: 4771527)
Fetched history for Will Hossack (ID: 4771400)
Max gameweek data available for is:  37
4771338 Ben Shankland GW 1 Pick 1
4771338 Ben Shankland GW 1 Pick 2
4771338 Ben Shankland GW 1 Pick 3
4771338 Ben Shankland GW 1 Pick 4
4771338 Ben Shankland GW 1 Pick 5
4771338 Ben Shankland GW 1 Pick 6
4771338 Ben Shankland GW 1 Pick 7
4771338 Ben Shankland GW 1 Pick 8
4771338 Ben Shankland GW 1 Pick 9
4771338 Ben Shankland GW 1 Pick 10
4771338 Ben Shankland GW 1 Pick 11
4771338 Ben Shankland GW 1 Pick 12
4771338 Ben Shankland GW 1 Pick 13
4771338 Ben Shankland GW 1 Pick 14
4771338 Ben Shankland GW 1 Pick 15
4771338 Ben Shankland GW 2 Pick 1
4771338 Ben Shankland GW 2 

In [11]:
fpl_picks['gw'].max()

np.float64(36.0)

# Get FPL Player current status

In [12]:
#Create DF from FPL API Request
headers={'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
get=requests.get("https://fantasy.premierleague.com/api/bootstrap-static/", headers=headers, timeout=5)
data=json.loads(get.text)

#player_raw = Summary stats for each player for the 2023/24 season
cols=list(data['elements'][0].keys())
player_raw = pd.DataFrame(data['elements'], columns = cols)
player = player_raw[['web_name','id','now_cost','status','news','chance_of_playing_next_round','chance_of_playing_this_round']]
player['season']=20252026
player['datetime_now'] = pd.to_datetime('now')

table_name = "player_status"
if __name__ == "__main__":
    db_creator.create_staging_table_then_insert_data(table_name, data=player)
    player_status = db_creator.table_to_df(table_name=table_name)

C:\Users\Chris\AppData\Local\Temp\ipykernel_12500\1914937267.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player['season']=20252026
C:\Users\Chris\AppData\Local\Temp\ipykernel_12500\1914937267.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player['datetime_now'] = pd.to_datetime('now')


In [13]:
finish=time.perf_counter()
print(f'Finished in {round(((finish-start)/60),2)} minute(s)')
#Finished in 11 minute(s)

Finished in 14.37 minute(s)


In [14]:
# Notification when file has finished running
from plyer import notification

notification.notify(
    title='Script Complete',
    message='Your file has finished running!',
    app_name='Jupyter Notebook',
    timeout=20  # notification stays for 20 seconds
)